In [ ]:
#läs in excel
#för varje row
#skapa filsökväg
#kör detect på filen
#om detect giltigt:
#spara ner bild plus GT

In [ ]:
import pandas as pd
from glob import glob
from pathlib import Path

import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

# Check MMDetection installation
import mmdet
print(mmdet.__version__)

# Check mmcv installation
from mmcv.ops import get_compiling_cuda_version, get_compiler_version
print(get_compiling_cuda_version())
print(get_compiler_version())

import mmcv
import matplotlib.pyplot as plt

In [ ]:
from mmcv import Config
cfg = Config.fromfile('/ceph/hpc/home/euerikl/projects/fastighet/models/configs/cascade_rcnn2.py')

In [ ]:
from mmdet.apis import inference_detector, init_detector
checkpoint = '/ceph/hpc/home/euerikl/projects/fastighet/models/cascade_rcnn_1/latest.pth'
model = init_detector(cfg, checkpoint, device='cuda:0')

In [9]:
imgs = glob('/ceph/hpc/scratch/user/euerikl/data/fastighet/miljonsetet/all_batches/10018814/*.tif')

for img in imgs:
    im = mmcv.imread(img)
    result = inference_detector(model, im)

/ceph/hpc/home/euerikl/.conda/envs/open-mmlab/lib/python3.7/site-packages/mmdet/datasets/utils.py:70: UserWarning: "ImageToTensor" pipeline is replaced by "DefaultFormatBundle" for batch inference. It is recommended to manually replace it in the test data pipeline in your config file.
  'data pipeline in your config file.', UserWarning)


In [ ]:
df_million = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/miljonsetet/miljon_testet_facit_enkel_FNR.txt', '\t', index_col=2, encoding='latin-1', dtype=str)
print(len(df_million))
df_million = df_million.drop_duplicates()
print(len(df_million))

In [ ]:
%%time

import os

total_number_of_instances = 0

parent_out_dir = '/home/erik/Riksarkivet/Projects/fastighet/data/htr_training_set_1_million_plus'
path_to_batches = '/home/erik/Riksarkivet/Projects/fastighet/data/miljonsetet/f'
batches = glob(os.path.join(path_to_batches, '**'))
batches.sort()
no_of_batches = 0

with open(os.path.join(parent_out_dir, 'gt_f.txt'), 'w') as f:

    for batch in batches:
        no_of_batches += 1
        batch_out_dir = os.path.join(parent_out_dir, Path(batch).name)
        batch_name = Path(batch).name
        
        os.mkdir(batch_out_dir)

        imgs = glob(os.path.join(batch, '**'))
        imgs.sort()
        number_of_inst = 0
        number_of_err = 0
        number_of_zeros = 0
        
        for img in imgs:
            img_name = Path(img).name.split('.')[0]
            try:
                row = df_million.loc[img_name]
                trakt = row['GTRAKT']
                block = row['GBLOCK']
                enhet = row['GENHET']

                im = mmcv.imread(img)
                result = inference_detector(model, im)
                if len(result[0]) == 1:
                    number_of_inst += 1
                    out_path = os.path.join(batch_out_dir, Path(img).name)
                    cropped_img = mmcv.imcrop(im, result[0][0][0:4])
                    mmcv.imwrite(cropped_img, out_path)
                    
                    img_path = os.path.join(Path(batch).name, Path(img).name)
                    f.write(img_path + '|' + trakt + ';' + block + ';' + enhet + '\n')
                elif len(result[0]) == 0:
                    number_of_zeros += 1

            except:
                number_of_err += 1

        print(batch_name + ':' + str(no_of_batches) + ':' + str(number_of_inst) + ':' + str(number_of_err) + ':' + str(number_of_zeros))
        total_number_of_instances += number_of_inst

print(total_number_of_instances)
